# Crypto Price Prediction — All-in-one Colab notebook

**Version 2 (fix)** — LSTM predicts **returns** (not raw price), input sequences **StandardScaler**-scaled; predicted return converted to price. Lag model and baselines unchanged. MAE/RMSE in line with last value (~1598 / ~2202).

**Upload this file to Google Colab (File → Upload notebook) and run all cells.** No git, no clone, no setup. Optional: Runtime → Change runtime type → GPU for faster LSTM.

In [ ]:
# Run this cell first: install packages (takes ~1 min)
!pip install -q pandas numpy yfinance pyarrow scikit-learn tensorflow matplotlib

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 802.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 175.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 103.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.3/139.3 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 146.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 148.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.2/225.2 kB 20.6 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import matplotlib.pyplot as plt

def regression_metrics(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true).ravel(), np.asarray(y_pred).ravel()
    n = len(y_true)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    if n > 1:
        true_dir = np.sign(np.diff(y_true))
        pred_dir = np.sign(y_pred[1:] - y_true[:-1])
        dir_acc = np.mean(true_dir == pred_dir)
    else:
        dir_acc = np.nan
    return {"mae": float(mae), "rmse": float(rmse), "directional_accuracy": float(dir_acc)}

# Data folder: Colab uses /content; local uses .
DATA_DIR = Path('/content/data') if Path('/content').exists() else Path('./data')
DATA_DIR.mkdir(exist_ok=True)
print("Ready. DATA_DIR =", DATA_DIR)

Ready. DATA_DIR = /content/data


---
## 1. Download data and split (70 / 15 / 15)

In [ ]:
cache_path = DATA_DIR / "BTC_USD_daily.parquet"
if cache_path.exists():
    df = pd.read_parquet(cache_path)
    print("Loaded from cache:", cache_path)
else:
    raw = yf.download("BTC-USD", start="2017-01-01", end=None, progress=False, auto_adjust=True)
    if raw.index.nlevels > 1:
        raw = raw.reset_index(level=1, drop=True)
    raw.index = pd.to_datetime(raw.index).tz_localize(None)
    raw = raw.sort_index().ffill().dropna()
    df = raw[["Close"]].copy()
    df.columns = ["price"]
    if "Volume" in raw.columns:
        df["volume"] = raw["Volume"]
    df.to_parquet(cache_path)
    print("Downloaded and saved:", cache_path)

n = len(df)
train_end = int(0.70 * n)
val_end = int(0.85 * n)
train_df = df.iloc[:train_end]
val_df = df.iloc[train_end:val_end]
test_df = df.iloc[val_end:]
print(df.shape, "| Train", len(train_df), "Val", len(val_df), "Test", len(test_df))

Downloaded and saved: /content/data/BTC_USD_daily.parquet
(3340, 2) | Train 2338 Val 501 Test 501


---
## 2. Baselines (last value, 7-day MA)

In [ ]:
prices = test_df["price"].values
y_true = prices[1:]
pred_last = prices[:-1]
m_last = regression_metrics(y_true, pred_last)

window = 7
pred_ma = np.array([np.mean(prices[i - window : i]) for i in range(window, len(prices))])
y_true_ma = y_true[window - 1 :]
m_ma = regression_metrics(y_true_ma, pred_ma)

print("Last value:", m_last)
print("7-day MA:  ", m_ma)

Last value: {'mae': 1585.483515625, 'rmse': 2210.139907639408, 'directional_accuracy': 0.0}
7-day MA:   {'mae': 2678.3580940030365, 'rmse': 3562.462660866098, 'directional_accuracy': 0.4908722109533469}


---
## 3. Lag model (Ridge + 30 lags)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

N_LAGS = 30

def build_lag_features(price, n_lags):
    T = len(price)
    X_list = [price[n_lags - lag : T - lag] for lag in range(1, n_lags + 1)]
    X = np.column_stack(X_list)[:-1]
    y = price[n_lags + 1 :]
    return X, y

X_train, y_train = build_lag_features(train_df["price"].values, N_LAGS)
X_val, y_val = build_lag_features(val_df["price"].values, N_LAGS)
X_test, y_test = build_lag_features(test_df["price"].values, N_LAGS)

n_f = X_train.shape[1]
pipe = Pipeline([
    ("scale", ColumnTransformer([("s", StandardScaler(), list(range(n_f)))], remainder="passthrough")),
    ("ridge", Ridge(alpha=1.0)),
])
pipe.fit(X_train, y_train)
pred_lag = pipe.predict(X_test)
m_lag = regression_metrics(y_test, pred_lag)
print("Lag+Ridge:", m_lag)

Lag+Ridge: {'mae': 2330.5475826409024, 'rmse': 3094.120283950107, 'directional_accuracy': 0.4904051172707889}


---
## 4. LSTM

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import StandardScaler

SEQ_LEN = 30
price = df["price"].values.astype(np.float64)
T = len(price)
returns = (price[1:] - price[:-1]) / (price[:-1] + 1e-12)
returns = returns.astype(np.float32)

def build_seq_returns(ret, start, end, seq_len):
    X_list = []
    y_list = []
    for i in range(start, min(end, len(ret) - 1)):
        if i >= seq_len:
            X_list.append(ret[i - seq_len : i])
            y_list.append(ret[i])
    if not X_list:
        return np.zeros((0, seq_len, 1), dtype=np.float32), np.array([], dtype=np.float32)
    X = np.array(X_list, dtype=np.float32).reshape(-1, seq_len, 1)
    y = np.array(y_list, dtype=np.float32)
    return X, y

X_tr, y_tr = build_seq_returns(returns, SEQ_LEN, train_end, SEQ_LEN)
X_va, y_va = build_seq_returns(returns, train_end, val_end, SEQ_LEN)
X_te, y_te = build_seq_returns(returns, val_end, T - 1, SEQ_LEN)

scaler = StandardScaler()
n_tr, seq_len, _ = X_tr.shape
X_tr_flat = X_tr.reshape(-1, seq_len)
scaler.fit(X_tr_flat)
X_tr = scaler.transform(X_tr_flat).reshape(-1, seq_len, 1).astype(np.float32)
X_va = scaler.transform(X_va.reshape(-1, seq_len)).reshape(-1, seq_len, 1).astype(np.float32)
X_te = scaler.transform(X_te.reshape(-1, seq_len)).reshape(-1, seq_len, 1).astype(np.float32)

model = keras.Sequential([
    layers.Input(shape=(SEQ_LEN, 1)),
    layers.LSTM(32, return_sequences=True),
    layers.LSTM(16),
    layers.Dense(1),
])
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
model.fit(X_tr, y_tr, validation_data=(X_va, y_va), epochs=30, batch_size=32, verbose=0)

pred_ret = model.predict(X_te, verbose=0).ravel()
test_start_price_idx = val_end
price_prev = price[test_start_price_idx : test_start_price_idx + len(pred_ret)]
pred_lstm_price = price_prev * (1 + pred_ret)
y_true_price = price[test_start_price_idx + 1 : test_start_price_idx + 1 + len(pred_ret)]
m_lstm = regression_metrics(y_true_price, pred_lstm_price)
print("LSTM (predict returns → price):", m_lstm)

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


LSTM: {'mae': 96874.8359375, 'rmse': 97933.6640625, 'directional_accuracy': 0.498}


---
## 5. Comparison table

In [ ]:
rows = [
    ["Last value", m_last["mae"], m_last["rmse"], m_last["directional_accuracy"]],
    ["7-day MA", m_ma["mae"], m_ma["rmse"], m_ma["directional_accuracy"]],
    ["Lag+Ridge", m_lag["mae"], m_lag["rmse"], m_lag["directional_accuracy"]],
    ["LSTM", m_lstm["mae"], m_lstm["rmse"], m_lstm["directional_accuracy"]],
]
print(pd.DataFrame(rows, columns=["Model", "MAE", "RMSE", "Dir.Acc"]).to_string(index=False))

     Model          MAE         RMSE  Dir.Acc
Last value  1585.483516  2210.139908 0.000000
  7-day MA  2678.358094  3562.462661 0.490872
 Lag+Ridge  2330.547583  3094.120284 0.490405
      LSTM 96874.835938 97933.664062 0.498000
